# Exploratory Data Analysis - Bank Transaction Dataset

Notebook ini berisi analisis eksploratif terhadap dataset transaksi bank untuk mengidentifikasi pola-pola mencurigakan dan anomali yang berpotensi sebagai fraud.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load Data dan Overview Awal

In [ ]:
df = pd.read_csv('../data/bank_transactions_data_2.csv')
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# cek missing values
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print('Tidak ada missing values')

In [ ]:
# cek duplikat
print(f'Jumlah duplikat: {df.duplicated().sum()}')
print(f'Unique TransactionID: {df["TransactionID"].nunique()} dari {len(df)} rows')

## 2. Distribusi Variabel Numerik

In [ ]:
num_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    
    # tambahin garis mean dan median
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='-', label=f'Median: {df[col].median():.2f}')
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# boxplot untuk lihat outlier
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col], vert=True)
    axes[i].set_title(col)

plt.suptitle('Boxplot Variabel Numerik', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# summary statistik per variabel numerik
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: range [{df[col].min():.2f}, {df[col].max():.2f}], IQR outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')

## 3. Distribusi Variabel Kategorikal

In [ ]:
cat_cols = ['TransactionType', 'Channel', 'CustomerOccupation', 'Location']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    if len(counts) > 15:
        counts = counts.head(15)
    counts.plot(kind='barh', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# cardinality tiap kolom kategorikal
for col in ['TransactionType', 'Channel', 'CustomerOccupation', 'Location', 'DeviceID', 'MerchantID']:
    print(f'{col}: {df[col].nunique()} unique values')

## 4. Analisis Temporal

In [ ]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])

df['TxnHour'] = df['TransactionDate'].dt.hour
df['TxnDayOfWeek'] = df['TransactionDate'].dt.dayofweek
df['TxnMonth'] = df['TransactionDate'].dt.month
df['DaysSincePrevTxn'] = (df['TransactionDate'] - df['PreviousTransactionDate']).dt.days

print(f"Rentang transaksi: {df['TransactionDate'].min()} s/d {df['TransactionDate'].max()}")
print(f"Rentang previous txn: {df['PreviousTransactionDate'].min()} s/d {df['PreviousTransactionDate'].max()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# distribusi jam transaksi
df['TxnHour'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Transaksi per Jam')
axes[0].set_xlabel('Jam')
axes[0].set_ylabel('Jumlah')

# distribusi hari
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = df['TxnDayOfWeek'].value_counts().sort_index()
day_counts.index = [day_names[i] for i in day_counts.index]
day_counts.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Transaksi per Hari')
axes[1].set_xlabel('Hari')

# distribusi DaysSincePrevTxn
axes[2].hist(df['DaysSincePrevTxn'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[2].set_title('Distribusi Days Since Previous Transaction')
axes[2].set_xlabel('Days')

plt.tight_layout()
plt.show()

## 5. Korelasi antar Variabel Numerik

In [ ]:
corr_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance', 'DaysSincePrevTxn']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# scatter plot: TransactionAmount vs AccountBalance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df['AccountBalance'], df['TransactionAmount'], alpha=0.3, s=10)
axes[0].set_xlabel('Account Balance')
axes[0].set_ylabel('Transaction Amount')
axes[0].set_title('Amount vs Balance')

axes[1].scatter(df['TransactionDuration'], df['TransactionAmount'], alpha=0.3, s=10)
axes[1].set_xlabel('Transaction Duration')
axes[1].set_ylabel('Transaction Amount')
axes[1].set_title('Amount vs Duration')

axes[2].scatter(df['LoginAttempts'], df['TransactionAmount'], alpha=0.3, s=10)
axes[2].set_xlabel('Login Attempts')
axes[2].set_ylabel('Transaction Amount')
axes[2].set_title('Amount vs Login Attempts')

plt.tight_layout()
plt.show()

## 6. Identifikasi Pola Anomali

In [ ]:
# transaksi dengan amount sangat tinggi relatif terhadap balance
df['AmountToBalanceRatio'] = df['TransactionAmount'] / (df['AccountBalance'] + 1)

print('Top 10 transaksi dengan rasio amount/balance tertinggi:')
high_ratio = df.nlargest(10, 'AmountToBalanceRatio')[['TransactionID', 'TransactionAmount', 'AccountBalance', 'AmountToBalanceRatio', 'LoginAttempts', 'Channel']]
high_ratio

In [ ]:
# transaksi dengan login attempts tinggi (potential brute force)
print(f'Distribusi LoginAttempts:')
print(df['LoginAttempts'].value_counts().sort_index())
print(f'\nTransaksi dengan LoginAttempts > 3: {len(df[df["LoginAttempts"] > 3])} ({len(df[df["LoginAttempts"] > 3])/len(df)*100:.1f}%)')

In [ ]:
# analisis per channel: apakah ada channel tertentu yang punya pola beda
channel_stats = df.groupby('Channel').agg(
    avg_amount=('TransactionAmount', 'mean'),
    median_amount=('TransactionAmount', 'median'),
    avg_duration=('TransactionDuration', 'mean'),
    avg_login_attempts=('LoginAttempts', 'mean'),
    count=('TransactionID', 'count')
).round(2)

channel_stats

In [ ]:
# transaksi di jam-jam tidak wajar (misal dini hari) dengan amount tinggi
late_night = df[df['TxnHour'].between(0, 5)]
normal_hours = df[~df['TxnHour'].between(0, 5)]

print(f'Transaksi dini hari (00:00-05:59): {len(late_night)} ({len(late_night)/len(df)*100:.1f}%)')
print(f'Avg amount dini hari: {late_night["TransactionAmount"].mean():.2f}')
print(f'Avg amount jam normal: {normal_hours["TransactionAmount"].mean():.2f}')
print(f'Avg login attempts dini hari: {late_night["LoginAttempts"].mean():.2f}')
print(f'Avg login attempts jam normal: {normal_hours["LoginAttempts"].mean():.2f}')

In [ ]:
# cek apakah ada account yang melakukan banyak transaksi dalam waktu singkat
acct_txn_count = df.groupby('AccountID').agg(
    txn_count=('TransactionID', 'count'),
    total_amount=('TransactionAmount', 'sum'),
    avg_amount=('TransactionAmount', 'mean'),
    unique_locations=('Location', 'nunique'),
    unique_devices=('DeviceID', 'nunique')
).sort_values('txn_count', ascending=False)

print('Accounts dengan transaksi terbanyak:')
acct_txn_count.head(10)

In [ ]:
# account yang pakai banyak device atau lokasi berbeda (potential account takeover)
multi_device = acct_txn_count[acct_txn_count['unique_devices'] > 1]
multi_location = acct_txn_count[acct_txn_count['unique_locations'] > 1]

print(f'Accounts dengan >1 device: {len(multi_device)} ({len(multi_device)/df["AccountID"].nunique()*100:.1f}%)')
print(f'Accounts dengan >1 lokasi: {len(multi_location)} ({len(multi_location)/df["AccountID"].nunique()*100:.1f}%)')

## 7. Ringkasan Temuan (Business Summary)

Berdasarkan analisis eksploratif di atas, berikut temuan utama:

**Data Overview:**
- Dataset berisi 2,512 transaksi dengan 16 atribut, mencakup informasi transaksi, akun, device, dan demografi customer.
- Tidak ditemukan missing values maupun duplikasi data.

**Distribusi dan Pola:**
- TransactionAmount memiliki distribusi right-skewed, mayoritas transaksi bernilai kecil dengan beberapa transaksi bernilai sangat tinggi.
- CustomerAge tersebar merata, menunjukkan dataset mencakup berbagai segmen usia.
- LoginAttempts mayoritas bernilai 1, namun ada sejumlah transaksi dengan login attempts tinggi yang perlu diwaspadai.

**Pola Anomali yang Teridentifikasi:**
- Beberapa transaksi memiliki rasio amount/balance yang sangat tinggi, mengindikasikan transaksi yang tidak proporsional terhadap saldo.
- Transaksi dengan multiple login attempts bisa mengindikasikan brute force atau unauthorized access.
- Perlu perhatian khusus pada account yang menggunakan multiple device atau bertransaksi dari banyak lokasi berbeda.

**Rekomendasi untuk Modeling:**
- Feature engineering dari temporal patterns (jam transaksi, hari, jarak antar transaksi) akan berguna.
- Rasio amount terhadap balance, login attempts, dan multi-device usage bisa jadi indikator kuat untuk anomaly detection.
- Pendekatan unsupervised (Isolation Forest, LOF) cocok karena dataset tidak memiliki label fraud eksplisit.